# Version 4:
bias estimator first, then maybe position estimating head later

In [ ]:
# === Persistence setup (must run before any plots/training) ===
# All artifacts go to /kaggle/working/ which Kaggle saves with the notebook version.
import os
import pickle
import torch
import matplotlib.pyplot as plt

ARTIFACTS_DIR = '/kaggle/working/stage_c_bis'   # models + histories
PLOT_DIR      = '/kaggle/working/plots'         # figures
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

_plot_counter = {'n': 0}

def save_current_fig(name):
    """Persist the current matplotlib figure as a PNG.
    Call BEFORE plt.show(). Idempotent on errors.
    """
    try:
        _plot_counter['n'] += 1
        n = _plot_counter['n']
        path = f"{PLOT_DIR}/{n:02d}_{name}.png"
        plt.gcf().savefig(path, dpi=120, bbox_inches='tight')
        print(f"  [saved plot] {path}")
    except Exception as e:
        print(f"  [WARN] could not save plot {name}: {e}")

def save_model_and_history(model, history, name):
    """Persist a model state_dict + a training history dict to ARTIFACTS_DIR."""
    try:
        torch.save(model.state_dict(), f'{ARTIFACTS_DIR}/model_{name}.pt')
        with open(f'{ARTIFACTS_DIR}/history_{name}.pkl', 'wb') as f:
            pickle.dump(history, f)
        print(f"  [saved] {ARTIFACTS_DIR}/model_{name}.pt + history_{name}.pkl")
    except Exception as e:
        print(f"  [WARN] could not save {name}: {e}")

print(f"Persistence ready. Plots -> {PLOT_DIR}, models -> {ARTIFACTS_DIR}")


## Synthetic data generatio

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def generate_synthetic_window(
    duration=30.0, dt=0.1, v0=10.0, yaw0=0.0,
    motion='circle', motion_params=None, seed=0,
):
    if motion_params is None: motion_params = {}
    t = np.arange(0, duration + dt/2, dt)
    N = len(t)
    rng = np.random.default_rng(seed)

    if motion == 'straight':
        ax_true = np.zeros(N); wz_true = np.zeros(N)

    elif motion == 'accel_oscillating':
        # Replaces 'accel_straight'. Sinusoidal ax, zero-mean over the window.
        amp    = motion_params.get('ax_amp', 0.6)
        period = motion_params.get('period', 15.0)  # fits ~2 cycles in 30s
        ax_true = amp * np.sin(2 * np.pi * t / period)
        wz_true = np.zeros(N)

    elif motion == 'circle':
        ax_true = np.zeros(N); wz_true = np.full(N, motion_params.get('wz', 0.05))

    elif motion == 'slalom':
        amp    = motion_params.get('wz_amp', 0.1)
        period = motion_params.get('period', 10.0)
        ax_true = np.zeros(N)
        wz_true = amp * np.sin(2 * np.pi * t / period)

    elif motion == 'random':
        # Force zero mean by detrending after generation.
        ax_max = motion_params.get('ax_max', 1.0)
        wz_max = motion_params.get('wz_max', 0.1)
        ax_raw = np.cumsum(rng.normal(0, 0.3, N)) * dt
        wz_raw = np.cumsum(rng.normal(0, 0.05, N)) * dt
        ax_raw -= ax_raw.mean()   # zero-mean over window
        wz_raw -= wz_raw.mean()
        ax_true = np.clip(ax_raw, -ax_max, ax_max)
        wz_true = np.clip(wz_raw, -wz_max, wz_max)

    else:
        raise ValueError(f"Unknown motion: {motion}")

    # ... rest unchanged (integrate forward, return window dict)
    x   = np.zeros(N); y   = np.zeros(N)
    v   = np.zeros(N); yaw = np.zeros(N)
    x[0], y[0], v[0], yaw[0] = 0.0, 0.0, v0, yaw0
    for i in range(N - 1):
        x[i+1]   = x[i]   + v[i] * np.cos(yaw[i]) * dt
        y[i+1]   = y[i]   + v[i] * np.sin(yaw[i]) * dt
        v[i+1]   = v[i]   + ax_true[i] * dt
        yaw[i+1] = yaw[i] + wz_true[i] * dt
    ay_true = v * wz_true
    return {
        't':  t, 'dt': np.full(N, dt),
        'x_gt': x, 'y_gt': y, 'v_gt': v, 'yaw_gt': yaw,
        'ax':   ax_true, 'ay': ay_true, 'wz': wz_true,
        'v0':   float(v0), 'yaw0': float(yaw0),
        'drive': f'synthetic_{motion}',
    }

### Sanity check 1a — visual

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Assuming generate_synthetic_window is already defined in your environment
configs = [
    ('straight',          {}),
    ('accel_oscillating', {'ax_amp': 0.6, 'period': 15.0}),
    ('circle',            {'wz': 0.05}),
    ('slalom',            {'wz_amp': 0.15, 'period': 8}),
    ('random',            {}),
]

# 1. Create a 1x5 subplot grid and set the titles from the configs
fig = make_subplots(
    rows=1, cols=len(configs),
    subplot_titles=[config[0] for config in configs]
)

# 2. Loop through and add traces to each subplot
for i, (motion, params) in enumerate(configs):
    w = generate_synthetic_window(motion=motion, motion_params=params, seed=1)
    
    col = i + 1  # Plotly grid columns are 1-indexed

    # Trajectory ('b-')
    fig.add_trace(
        go.Scatter(
            x=w['x_gt'], y=w['y_gt'],
            mode='lines',
            line=dict(color='blue'),
            name='trajectory',
            legendgroup='traj',       # Groups items in the legend
            showlegend=(i == 0)       # Prevents duplicate legend entries
        ),
        row=1, col=col
    )

    # Start point ('go')
    fig.add_trace(
        go.Scatter(
            x=[w['x_gt'][0]], y=[w['y_gt'][0]],
            mode='markers',
            marker=dict(color='green', symbol='circle', size=8),
            name='start',
            legendgroup='start',
            showlegend=(i == 0)
        ),
        row=1, col=col
    )

    # End point ('rx')
    fig.add_trace(
        go.Scatter(
            x=[w['x_gt'][-1]], y=[w['y_gt'][-1]],
            mode='markers',
            marker=dict(color='red', symbol='x', size=8),
            name='end',
            legendgroup='end',
            showlegend=(i == 0)
        ),
        row=1, col=col
    )

    # Set aspect ratio to 'equal' (link the y-axis scale to its corresponding x-axis)
    x_axis_ref = f"x{col}" if col > 1 else "x"
    fig.update_yaxes(scaleanchor=x_axis_ref, scaleratio=1, row=1, col=col)

# 3. Replicate styling: tight layout, figure size, and subtle grid lines
fig.update_layout(
    width=1500,  # Scales similarly to Matplotlib's figsize=(20, ...)
    height=400,  # Scales similarly to Matplotlib's figsize=(..., 4)
    plot_bgcolor='white',
    margin=dict(l=20, r=20, t=40, b=20) # tight_layout equivalent
)

# Apply grid styling (equivalent to alpha=0.3)
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='rgba(0,0,0,0.1)', zeroline=False)
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='rgba(0,0,0,0.1)', zeroline=False)

save_current_fig('sanity_1a_motion_paths')
fig.show()

In [ ]:
w = generate_synthetic_window(motion='circle',
                              motion_params={'wz': 0.05}, seed=1)

# Re-integrate using the IMU stored in the window
N = len(w['t'])
xx, yy, vv, yy_yaw = np.zeros(N), np.zeros(N), np.zeros(N), np.zeros(N)
xx[0], yy[0], vv[0], yy_yaw[0] = 0.0, 0.0, w['v0'], w['yaw0']
for i in range(N - 1):
    xx[i+1]     = xx[i]     + vv[i] * np.cos(yy_yaw[i]) * w['dt'][i]
    yy[i+1]     = yy[i]     + vv[i] * np.sin(yy_yaw[i]) * w['dt'][i]
    vv[i+1]     = vv[i]     + w['ax'][i] * w['dt'][i]
    yy_yaw[i+1] = yy_yaw[i] + w['wz'][i] * w['dt'][i]

err_pos = np.max(np.sqrt((xx - w['x_gt'])**2 + (yy - w['y_gt'])**2))
err_v   = np.max(np.abs(vv - w['v_gt']))
err_yaw = np.max(np.abs(yy_yaw - w['yaw_gt']))
print(f"max position error: {err_pos:.2e} m")
print(f"max velocity error: {err_v:.2e} m/s")
print(f"max yaw error:      {err_yaw:.2e} rad")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


configs = [
    ('straight',          {}),
    ('accel_oscillating', {'ax_amp': 0.6, 'period': 15.0}),
    ('circle',            {'wz': 0.05}),
    ('slalom',            {'wz_amp': 0.15, 'period': 8}),
    ('random',            {}),
]

fig, axes = plt.subplots(len(configs), 5, figsize=(22, 3.2 * len(configs)))

for row, (motion, params) in enumerate(configs):
    w = generate_synthetic_window(motion=motion, motion_params=params, seed=1)
    t = w['t']

    axes[row, 0].plot(t, w['ax'], 'C0-')
    axes[row, 0].set_ylabel(f"{motion}\nax (m/s²)"); axes[row, 0].grid(alpha=0.3)

    axes[row, 1].plot(t, w['ay'], 'C1-')
    axes[row, 1].set_ylabel("ay (m/s²)"); axes[row, 1].grid(alpha=0.3)

    axes[row, 2].plot(t, w['wz'], 'C2-')
    axes[row, 2].set_ylabel("wz (rad/s)"); axes[row, 2].grid(alpha=0.3)

    axes[row, 3].plot(t, w['v_gt'],   'C3-', label='v')
    axes[row, 3].plot(t, w['yaw_gt'], 'C4-', label='yaw')
    axes[row, 3].set_ylabel("state"); axes[row, 3].legend(fontsize=8)
    axes[row, 3].grid(alpha=0.3)

    axes[row, 4].plot(w['x_gt'], w['y_gt'], 'k-')
    axes[row, 4].plot(w['x_gt'][0],  w['y_gt'][0],  'go')
    axes[row, 4].plot(w['x_gt'][-1], w['y_gt'][-1], 'rx')
    axes[row, 4].set_aspect('equal'); axes[row, 4].grid(alpha=0.3)
    axes[row, 4].set_ylabel("xy")

for ax in axes[-1]:
    ax.set_xlabel("t (s)" if ax is not axes[-1, 4] else "x (m)")

plt.tight_layout(); save_current_fig('sanity_2_imu_traces'); plt.show()

## GRU + Integrator

In [ ]:
import torch
import torch.nn as nn

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


class BiasGRU(nn.Module):
    def __init__(self, hidden=32, n_layers=1):
        super().__init__()
        self.gru  = nn.GRU(input_size=6, hidden_size=hidden,
                           num_layers=n_layers, batch_first=True)
        self.head = nn.Linear(hidden, 2)
        # Small (NOT zero) weights so initial bias predictions are tiny
        # but gradients can flow back into the GRU.
        nn.init.normal_(self.head.weight, std=1e-3)
        nn.init.zeros_(self.head.bias)

    def forward(self, imu_seq, state_seq):
        x = torch.cat([imu_seq, state_seq], dim=-1)
        h, _ = self.gru(x)
        bias = self.head(h)
        return bias


class GreyBoxPINN(nn.Module):
    def __init__(self, hidden=32):
        super().__init__()
        self.bias_net = BiasGRU(hidden=hidden)

    def forward(self, ax, ay, wz, dt, v0, yaw0):
        B, T = ax.shape

        # GRU pass over the whole sequence
        v0_seq     = v0.unsqueeze(1).expand(B, T)
        sin_yaw0_s = torch.sin(yaw0).unsqueeze(1).expand(B, T)
        cos_yaw0_s = torch.cos(yaw0).unsqueeze(1).expand(B, T)
        imu_seq    = torch.stack([ax, ay, wz], dim=-1)
        state_seq  = torch.stack([v0_seq, sin_yaw0_s, cos_yaw0_s], dim=-1)
        bias_seq   = self.bias_net(imu_seq, state_seq)   # (B, T, 2)

        # Build trajectory step by step using LISTS (not in-place writes)
        x_list   = [torch.zeros(B, device=ax.device)]
        y_list   = [torch.zeros(B, device=ax.device)]
        v_list   = [v0]
        yaw_list = [yaw0]

        for i in range(T - 1):
            b_a = bias_seq[:, i, 0]
            b_w = bias_seq[:, i, 1]
            ax_clean = ax[:, i] - b_a
            wz_clean = wz[:, i] - b_w

            x_next   = x_list[-1]   + v_list[-1] * torch.cos(yaw_list[-1]) * dt[:, i]
            y_next   = y_list[-1]   + v_list[-1] * torch.sin(yaw_list[-1]) * dt[:, i]
            v_next   = v_list[-1]   + ax_clean * dt[:, i]
            yaw_next = yaw_list[-1] + wz_clean * dt[:, i]

            x_list.append(x_next)
            y_list.append(y_next)
            v_list.append(v_next)
            yaw_list.append(yaw_next)

        x   = torch.stack(x_list,   dim=1)
        y   = torch.stack(y_list,   dim=1)
        v   = torch.stack(v_list,   dim=1)
        yaw = torch.stack(yaw_list, dim=1)

        return x, y, v, yaw, bias_seq

In [ ]:
model = GreyBoxPINN().to(DEVICE)

# Build a batch from synthetic windows
windows = [
    generate_synthetic_window(motion='circle',
                              motion_params={'wz': 0.05}, seed=1),
    generate_synthetic_window(motion='slalom',
                              motion_params={'wz_amp': 0.1, 'period': 10},
                              seed=2),
]

def stack_batch(windows):
    ax  = torch.tensor(np.stack([w['ax']  for w in windows]),
                       dtype=torch.float32, device=DEVICE)
    ay  = torch.tensor(np.stack([w['ay']  for w in windows]),
                       dtype=torch.float32, device=DEVICE)
    wz  = torch.tensor(np.stack([w['wz']  for w in windows]),
                       dtype=torch.float32, device=DEVICE)
    dt  = torch.tensor(np.stack([w['dt']  for w in windows]),
                       dtype=torch.float32, device=DEVICE)
    v0  = torch.tensor([w['v0']   for w in windows],
                       dtype=torch.float32, device=DEVICE)
    yaw0= torch.tensor([w['yaw0'] for w in windows],
                       dtype=torch.float32, device=DEVICE)
    return ax, ay, wz, dt, v0, yaw0

ax_b, ay_b, wz_b, dt_b, v0_b, yaw0_b = stack_batch(windows)
x_p, y_p, v_p, yaw_p, bias_p = model(ax_b, ay_b, wz_b, dt_b, v0_b, yaw0_b)

print(f"ax shape:    {ax_b.shape}    (expected (2, 301))")
print(f"x_p shape:   {x_p.shape}     (expected (2, 301))")
print(f"bias shape:  {bias_p.shape}  (expected (2, 301, 2))")
print(f"all finite:  {torch.isfinite(x_p).all().item()}")
print(f"\nInitial bias prediction (should be ~0 due to zero init):")
print(f"  b_a range: [{bias_p[:,:,0].min():.4f}, {bias_p[:,:,0].max():.4f}]")
print(f"  b_w range: [{bias_p[:,:,1].min():.4f}, {bias_p[:,:,1].max():.4f}]")

In [ ]:
model = GreyBoxPINN().to(DEVICE)
model.eval()

w = generate_synthetic_window(motion='circle',
                              motion_params={'wz': 0.05}, seed=1)
ax_b, ay_b, wz_b, dt_b, v0_b, yaw0_b = stack_batch([w])

with torch.no_grad():
    x_p, y_p, v_p, yaw_p, _ = model(ax_b, ay_b, wz_b, dt_b, v0_b, yaw0_b)

x_p, y_p = x_p[0].cpu().numpy(), y_p[0].cpu().numpy()

# Compare to ground truth (since bias=0 and IMU is clean, model output
# should match GT to floating-point precision)
err = np.max(np.sqrt((x_p - w['x_gt'])**2 + (y_p - w['y_gt'])**2))
print(f"max position error vs ground truth (bias=0, clean IMU): {err:.2e} m")

In [ ]:
model = GreyBoxPINN().to(DEVICE)
ax_b, ay_b, wz_b, dt_b, v0_b, yaw0_b = stack_batch(windows)
x_gt = torch.tensor(np.stack([w['x_gt'] for w in windows]),
                    dtype=torch.float32, device=DEVICE)
y_gt = torch.tensor(np.stack([w['y_gt'] for w in windows]),
                    dtype=torch.float32, device=DEVICE)

x_p, y_p, _, _, _ = model(ax_b, ay_b, wz_b, dt_b, v0_b, yaw0_b)
loss = ((x_p - x_gt)**2 + (y_p - y_gt)**2).mean()
loss.backward()

gru_grad_norm = sum(p.grad.norm().item()**2
                    for p in model.bias_net.gru.parameters()
                    if p.grad is not None) ** 0.5
head_grad_norm = sum(p.grad.norm().item()**2
                     for p in model.bias_net.head.parameters()
                     if p.grad is not None) ** 0.5

print(f"loss:               {loss.item():.4f}")
print(f"GRU gradient norm:  {gru_grad_norm:.6f}  (should be > 0)")
print(f"head gradient norm: {head_grad_norm:.6f}  (should be > 0)")

## training

In [ ]:
# --- 4.1 Build a varied training and validation set ---

def make_dataset(n_windows, seed_base=0):
    """Generate a mix of motion types with varied initial conditions."""
    rng = np.random.default_rng(seed_base)
    windows = []
    motion_types = ['straight', 'accel_oscillating', 'circle', 'slalom', 'random']
    for i in range(n_windows):
        motion = motion_types[i % len(motion_types)]
        v0   = float(rng.uniform(5.0, 20.0))
        yaw0 = float(rng.uniform(-np.pi, np.pi))
        if motion == 'accel_straight':
            params = {'ax': float(rng.uniform(-1.0, 1.0))}
        elif motion == 'accel_oscillating':
            params = {'ax_amp': float(rng.uniform(0.3, 0.8)), 'period': float(rng.uniform(10.0, 20.0))}
        elif motion == 'circle':
            params = {'wz': float(rng.choice([-1, 1]) * rng.uniform(0.02, 0.10))}
        elif motion == 'slalom':
            params = {'wz_amp': float(rng.uniform(0.05, 0.15)),
                      'period': float(rng.uniform(6.0, 12.0))}
        else:
            params = {}
        w = generate_synthetic_window(motion=motion, motion_params=params,
                                      v0=v0, yaw0=yaw0, seed=seed_base + i)
        windows.append(w)
    return windows

train_windows = make_dataset(80,  seed_base=0)
val_windows   = make_dataset(20,  seed_base=10_000)
print(f"Train: {len(train_windows)}  Val: {len(val_windows)}")

In [ ]:
def train_stage_a(model, train_windows, val_windows,
                  n_epochs=600, batch_size=32, lr=1e-3,
                  log_every=20):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    # Cosine LR schedule: lr smoothly decays from 1e-3 to ~0
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)

    history = {'epoch': [], 'train_loss': [], 'val_drift_30s': [],
               'val_bias_a': [], 'val_bias_w': []}

    # Pre-stack the full train set once for smooth monitoring later
    ax_T, ay_T, wz_T, dt_T, v0_T, yaw0_T = stack_batch(train_windows)
    x_gt_T = torch.tensor(np.stack([w['x_gt'] for w in train_windows]),
                          dtype=torch.float32, device=DEVICE)
    y_gt_T = torch.tensor(np.stack([w['y_gt'] for w in train_windows]),
                          dtype=torch.float32, device=DEVICE)

    for ep in range(n_epochs):
        model.train()
        # FULL PASS through train set every epoch (multiple mini-batches)
        idxs = np.random.permutation(len(train_windows))
        for start in range(0, len(train_windows), batch_size):
            batch_idx = idxs[start:start + batch_size]
            batch = [train_windows[i] for i in batch_idx]
            ax_b, ay_b, wz_b, dt_b, v0_b, yaw0_b = stack_batch(batch)
            x_gt = torch.tensor(np.stack([w['x_gt'] for w in batch]),
                                dtype=torch.float32, device=DEVICE)
            y_gt = torch.tensor(np.stack([w['y_gt'] for w in batch]),
                                dtype=torch.float32, device=DEVICE)
            x_p, y_p, _, _, _ = model(ax_b, ay_b, wz_b, dt_b, v0_b, yaw0_b)
            loss = ((x_p - x_gt)**2 + (y_p - y_gt)**2).mean()
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        scheduler.step()

        if ep % log_every == 0 or ep == n_epochs - 1:
            model.eval()
            with torch.no_grad():
                # SMOOTH monitoring: loss on full train set, not just last batch
                x_p, y_p, _, _, bias_p = model(ax_T, ay_T, wz_T, dt_T, v0_T, yaw0_T)
                train_loss = ((x_p - x_gt_T)**2 + (y_p - y_gt_T)**2).mean().item()

                val_drifts, val_b_a, val_b_w = [], [], []
                for vw in val_windows:
                    ax_v, ay_v, wz_v, dt_v, v0_v, yaw0_v = stack_batch([vw])
                    x_v, y_v, _, _, bias_v = model(ax_v, ay_v, wz_v, dt_v, v0_v, yaw0_v)
                    drift = float(torch.sqrt(
                        (x_v[0, -1] - vw['x_gt'][-1])**2 +
                        (y_v[0, -1] - vw['y_gt'][-1])**2))
                    val_drifts.append(drift)
                    val_b_a.append(float(bias_v[:,:,0].mean()))
                    val_b_w.append(float(bias_v[:,:,1].mean()))
            history['epoch'].append(ep)
            history['train_loss'].append(train_loss)
            history['val_drift_30s'].append(float(np.mean(val_drifts)))
            history['val_bias_a'].append(float(np.mean(np.abs(val_b_a))))
            history['val_bias_w'].append(float(np.mean(np.abs(val_b_w))))
            print(f"  ep {ep:4d}  train_loss={train_loss:8.4f}  "
                  f"val_drift_30s={np.mean(val_drifts):6.3f}m  "
                  f"|b_a|={np.mean(np.abs(val_b_a)):.5f}  "
                  f"|b_w|={np.mean(np.abs(val_b_w)):.5f}  "
                  f"lr={opt.param_groups[0]['lr']:.5f}")
    return history


torch.manual_seed(0); np.random.seed(0)
model_a = GreyBoxPINN().to(DEVICE)
history_a = train_stage_a(model_a, train_windows, val_windows)

# Persist Stage A artifacts
save_model_and_history(model_a, history_a, 'stage_a')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history_a['epoch'], history_a['train_loss'])
axes[0].set_yscale('log'); axes[0].set_xlabel('epoch')
axes[0].set_ylabel('train MSE (m²)'); axes[0].grid(alpha=0.3)
axes[0].set_title('Training loss')

axes[1].plot(history_a['epoch'], history_a['val_drift_30s'])
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('mean drift @ 30s (m)')
axes[1].grid(alpha=0.3); axes[1].set_title('Validation drift')

axes[2].plot(history_a['epoch'], history_a['val_bias_a'], label='|b_a|')
axes[2].plot(history_a['epoch'], history_a['val_bias_w'], label='|b_w|')
axes[2].set_yscale('log'); axes[2].set_xlabel('epoch'); axes[2].legend()
axes[2].grid(alpha=0.3); axes[2].set_title('Val mean |predicted bias|')

plt.tight_layout(); save_current_fig('stage_a_loss_curves'); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
model_a.eval()
for ax_plot, vw in zip(axes, val_windows[:4]):
    ax_v, ay_v, wz_v, dt_v, v0_v, yaw0_v = stack_batch([vw])
    with torch.no_grad():
        x_v, y_v, _, _, _ = model_a(ax_v, ay_v, wz_v, dt_v, v0_v, yaw0_v)
    x_p = x_v[0].cpu().numpy(); y_p = y_v[0].cpu().numpy()
    ax_plot.plot(vw['x_gt'], vw['y_gt'], 'k-',  lw=2, label='GT')
    ax_plot.plot(x_p,        y_p,         'r--', lw=1, label='PINN')
    ax_plot.set_aspect('equal'); ax_plot.grid(alpha=0.3); ax_plot.legend(fontsize=8)
plt.tight_layout(); save_current_fig('stage_a_trajectories'); plt.show()

## Training stage B — constant bias only

In [ ]:
# --- 5.1 Bias injection ---

NOISE_PARAMS_B = dict(
    sigma_b0_a=0.10,    # std of constant accel bias drawn per window (m/s²)
    sigma_b0_w=0.015,   # std of constant gyro bias drawn per window (rad/s)
)

def inject_constant_bias(window, noise_params, seed):
    """
    Add a constant bias (drawn fresh per call) to ax and wz.
    No white noise, no random walk — just constant offsets.
    Returns dict with noisy ax/ay/wz and the realized biases for diagnostics.
    """
    rng = np.random.default_rng(seed)
    b0_ax = rng.normal(0.0, noise_params['sigma_b0_a'])
    b0_wz = rng.normal(0.0, noise_params['sigma_b0_w'])
    return {
        'ax':    window['ax'] + b0_ax,
        'ay':    window['ay'],          # ay not biased here; KITTI parity later
        'wz':    window['wz'] + b0_wz,
        'b0_ax': float(b0_ax),
        'b0_wz': float(b0_wz),
    }


def stack_batch_noisy(windows, noise_params, base_seed):
    """Stack a batch using noisy IMU drawn fresh from base_seed."""
    noisy = [inject_constant_bias(w, noise_params, base_seed + i)
             for i, w in enumerate(windows)]
    ax = torch.tensor(np.stack([n['ax'] for n in noisy]),
                      dtype=torch.float32, device=DEVICE)
    ay = torch.tensor(np.stack([n['ay'] for n in noisy]),
                      dtype=torch.float32, device=DEVICE)
    wz = torch.tensor(np.stack([n['wz'] for n in noisy]),
                      dtype=torch.float32, device=DEVICE)
    dt = torch.tensor(np.stack([w['dt'] for w in windows]),
                      dtype=torch.float32, device=DEVICE)
    v0 = torch.tensor([w['v0']   for w in windows],
                      dtype=torch.float32, device=DEVICE)
    yaw0 = torch.tensor([w['yaw0'] for w in windows],
                        dtype=torch.float32, device=DEVICE)
    b_a_true = torch.tensor([n['b0_ax'] for n in noisy],
                            dtype=torch.float32, device=DEVICE)
    b_w_true = torch.tensor([n['b0_wz'] for n in noisy],
                            dtype=torch.float32, device=DEVICE)
    return ax, ay, wz, dt, v0, yaw0, b_a_true, b_w_true


# Quick sanity: inject bias, run classical DR, confirm drift is ~tens of meters
def classical_dr(window, noisy):
    """Forward-Euler integration on noisy IMU, no bias correction."""
    N = len(window['t'])
    x, y, v, yaw = np.zeros(N), np.zeros(N), np.zeros(N), np.zeros(N)
    v[0], yaw[0] = window['v0'], window['yaw0']
    for i in range(N - 1):
        dt = window['dt'][i]
        x[i+1]   = x[i]   + v[i] * np.cos(yaw[i]) * dt
        y[i+1]   = y[i]   + v[i] * np.sin(yaw[i]) * dt
        v[i+1]   = v[i]   + noisy['ax'][i] * dt
        yaw[i+1] = yaw[i] + noisy['wz'][i] * dt
    return x, y

# Sanity check: classical DR drift on a few val windows
np.random.seed(42)
sample_drifts = []
for vw in val_windows[:10]:
    for s in range(5):
        noisy = inject_constant_bias(vw, NOISE_PARAMS_B, seed=1000 + s)
        x_dr, y_dr = classical_dr(vw, noisy)
        drift = np.sqrt((x_dr[-1] - vw['x_gt'][-1])**2 +
                        (y_dr[-1] - vw['y_gt'][-1])**2)
        sample_drifts.append(drift)
print(f"Classical DR drift on val (constant bias only):")
print(f"  mean: {np.mean(sample_drifts):.2f} m,  median: {np.median(sample_drifts):.2f} m,  "
      f"p95: {np.percentile(sample_drifts, 95):.2f} m")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Pick one window for visualization
w = generate_synthetic_window(motion='circle',
                              motion_params={'wz': 0.05},
                              v0=10.0, yaw0=0.0, seed=1)
t = w['t']
N = len(t)

# Noise parameters from Stage B (constant bias only)
sigma_b0_a = 0.10      # constant accel bias std (m/s²)
sigma_b0_w = 0.015     # constant gyro bias std (rad/s)
# Stage C adds these:
sigma_a    = 0.05      # white noise on accel (m/s²)
sigma_w    = 0.01      # white noise on gyro (rad/s)
sigma_brwa = 0.001     # bias random walk for accel (m/s²/√s)
sigma_brww = 0.0002    # bias random walk for gyro (rad/s/√s)

rng = np.random.default_rng(42)
dt_val = 0.1

# Draw the four noise components separately so we can visualize them in isolation
b0_ax  = rng.normal(0.0, sigma_b0_a)                                # one number
b0_wz  = rng.normal(0.0, sigma_b0_w)                                # one number
white_ax = rng.normal(0.0, sigma_a, N)                              # per-step
white_wz = rng.normal(0.0, sigma_w, N)
brw_ax = np.cumsum(rng.normal(0.0, sigma_brwa * np.sqrt(dt_val), N))   # cumulative
brw_wz = np.cumsum(rng.normal(0.0, sigma_brww * np.sqrt(dt_val), N))

# The complete noisy signal is clean + ALL of these
noisy_ax_full = w['ax'] + b0_ax + white_ax + brw_ax
noisy_wz_full = w['wz'] + b0_wz + white_wz + brw_wz

# --- Plot accel side ---
fig, axes = plt.subplots(2, 5, figsize=(22, 8))

axes[0, 0].plot(t, w['ax'], 'k-', lw=2)
axes[0, 0].set_title('Clean ax'); axes[0, 0].grid(alpha=0.3)
axes[0, 0].set_ylabel('m/s²'); axes[0, 0].set_xlabel('t (s)')

axes[0, 1].axhline(b0_ax, color='C0', lw=2)
axes[0, 1].set_title(f'Constant bias (b₀={b0_ax:+.3f})')
axes[0, 1].grid(alpha=0.3); axes[0, 1].set_xlabel('t (s)')
axes[0, 1].set_ylim(-3*sigma_b0_a, 3*sigma_b0_a)

axes[0, 2].plot(t, white_ax, 'C1-', lw=0.7)
axes[0, 2].set_title(f'White noise (σ={sigma_a})')
axes[0, 2].grid(alpha=0.3); axes[0, 2].set_xlabel('t (s)')

axes[0, 3].plot(t, brw_ax, 'C2-', lw=1.5)
axes[0, 3].set_title(f'Random walk (σ={sigma_brwa}/√s)')
axes[0, 3].grid(alpha=0.3); axes[0, 3].set_xlabel('t (s)')

axes[0, 4].plot(t, w['ax'], 'k-', lw=2, label='clean')
axes[0, 4].plot(t, noisy_ax_full, 'r-', lw=0.7, alpha=0.7, label='noisy')
axes[0, 4].set_title('Clean vs total noisy ax')
axes[0, 4].grid(alpha=0.3); axes[0, 4].legend(); axes[0, 4].set_xlabel('t (s)')

# --- Plot gyro side ---
axes[1, 0].plot(t, w['wz'], 'k-', lw=2)
axes[1, 0].set_title('Clean wz'); axes[1, 0].grid(alpha=0.3)
axes[1, 0].set_ylabel('rad/s'); axes[1, 0].set_xlabel('t (s)')

axes[1, 1].axhline(b0_wz, color='C0', lw=2)
axes[1, 1].set_title(f'Constant bias (b₀={b0_wz:+.4f})')
axes[1, 1].grid(alpha=0.3); axes[1, 1].set_xlabel('t (s)')
axes[1, 1].set_ylim(-3*sigma_b0_w, 3*sigma_b0_w)

axes[1, 2].plot(t, white_wz, 'C1-', lw=0.7)
axes[1, 2].set_title(f'White noise (σ={sigma_w})')
axes[1, 2].grid(alpha=0.3); axes[1, 2].set_xlabel('t (s)')

axes[1, 3].plot(t, brw_wz, 'C2-', lw=1.5)
axes[1, 3].set_title(f'Random walk (σ={sigma_brww}/√s)')
axes[1, 3].grid(alpha=0.3); axes[1, 3].set_xlabel('t (s)')

axes[1, 4].plot(t, w['wz'], 'k-', lw=2, label='clean')
axes[1, 4].plot(t, noisy_wz_full, 'r-', lw=0.7, alpha=0.7, label='noisy')
axes[1, 4].set_title('Clean vs total noisy wz')
axes[1, 4].grid(alpha=0.3); axes[1, 4].legend(); axes[1, 4].set_xlabel('t (s)')

plt.tight_layout(); save_current_fig('stage_b_classical_dr_check'); plt.show()

print(f"Magnitude check:")
print(f"  Clean ax range:   [{w['ax'].min():+.3f}, {w['ax'].max():+.3f}]")
print(f"  Constant bias:     {b0_ax:+.3f}")
print(f"  White noise std:   {white_ax.std():.3f}  (target {sigma_a})")
print(f"  RW final value:    {brw_ax[-1]:+.4f}  (expected std at t=30s: "
      f"{sigma_brwa*np.sqrt(30):.4f})")
print()
print(f"  Clean wz range:   [{w['wz'].min():+.4f}, {w['wz'].max():+.4f}]")
print(f"  Constant bias:     {b0_wz:+.4f}")
print(f"  White noise std:   {white_wz.std():.4f}  (target {sigma_w})")
print(f"  RW final value:    {brw_wz[-1]:+.5f}  (expected std at t=30s: "
      f"{sigma_brww*np.sqrt(30):.5f})")

In [ ]:
# --- 5.2 Training stage B ---

def train_stage_b(model, train_windows, val_windows, noise_params,
                  n_epochs=600, batch_size=32, lr=1e-3, log_every=20,
                  lambda_smooth=100.0):
    """
    Training with per-window noise injection and bias smoothness regularizer.

    The smoothness term penalizes sudden changes in predicted bias between
    consecutive timesteps, encouraging the network to output a slowly-varying
    bias estimate (which matches the true physics — bias evolves via random
    walk, not by jumping around).
    """
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)
    history = {'epoch': [], 'train_loss': [], 'pos_loss': [], 'smooth_loss': [],
               'val_drift_30s': [], 'bias_corr_a': [], 'bias_corr_w': []}

    for ep in range(n_epochs):
        model.train()
        idxs = np.random.permutation(len(train_windows))
        epoch_seed = 1_000_000 + ep * 10_000

        # Track loss components for the printout
        last_pos, last_smooth = 0.0, 0.0

        for start in range(0, len(train_windows), batch_size):
            batch_idx = idxs[start:start + batch_size]
            batch = [train_windows[i] for i in batch_idx]
            ax_b, ay_b, wz_b, dt_b, v0_b, yaw0_b, _, _ = stack_batch_noisy(
                batch, noise_params, epoch_seed + start)
            x_gt = torch.tensor(np.stack([w['x_gt'] for w in batch]),
                                dtype=torch.float32, device=DEVICE)
            y_gt = torch.tensor(np.stack([w['y_gt'] for w in batch]),
                                dtype=torch.float32, device=DEVICE)

            x_p, y_p, _, _, bias_seq = model(ax_b, ay_b, wz_b, dt_b, v0_b, yaw0_b)

            # Position loss: how far off is the predicted trajectory?
            pos_loss = ((x_p - x_gt)**2 + (y_p - y_gt)**2).mean()

            # Smoothness loss: penalize timestep-to-timestep bias jumps.
            # Bias should evolve slowly (random walk), not jitter wildly.
            smooth_loss = ((bias_seq[:, 1:, :] - bias_seq[:, :-1, :])**2).mean()

            loss = pos_loss + lambda_smooth * smooth_loss

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            last_pos = float(pos_loss)
            last_smooth = float(smooth_loss)

        scheduler.step()

        if ep % log_every == 0 or ep == n_epochs - 1:
            model.eval()
            with torch.no_grad():
                val_drifts, true_bas, pred_bas, true_bws, pred_bws = [], [], [], [], []
                for vw in val_windows:
                    for s in range(3):
                        ax_v, ay_v, wz_v, dt_v, v0_v, yaw0_v, b_a_t, b_w_t = \
                            stack_batch_noisy([vw], noise_params,
                                              base_seed=900_000 + s * 100)
                        x_v, y_v, _, _, bias_v = model(
                            ax_v, ay_v, wz_v, dt_v, v0_v, yaw0_v)
                        drift = float(torch.sqrt(
                            (x_v[0, -1] - vw['x_gt'][-1])**2 +
                            (y_v[0, -1] - vw['y_gt'][-1])**2))
                        val_drifts.append(drift)
                        true_bas.append(float(b_a_t[0]))
                        pred_bas.append(float(bias_v[:,:,0].mean()))
                        true_bws.append(float(b_w_t[0]))
                        pred_bws.append(float(bias_v[:,:,1].mean()))
                corr_a = np.corrcoef(true_bas, pred_bas)[0, 1]
                corr_w = np.corrcoef(true_bws, pred_bws)[0, 1]

            history['epoch'].append(ep)
            history['train_loss'].append(last_pos + lambda_smooth * last_smooth)
            history['pos_loss'].append(last_pos)
            history['smooth_loss'].append(last_smooth)
            history['val_drift_30s'].append(float(np.mean(val_drifts)))
            history['bias_corr_a'].append(float(corr_a))
            history['bias_corr_w'].append(float(corr_w))
            print(f"  ep {ep:4d}  pos={last_pos:7.2f}  "
                  f"smooth={last_smooth:.5f}  "
                  f"val_drift={np.mean(val_drifts):6.2f}m  "
                  f"corr(b_a)={corr_a:+.3f}  corr(b_w)={corr_w:+.3f}  "
                  f"lr={opt.param_groups[0]['lr']:.5f}")
    return history


# Train fresh model from scratch (don't reuse Stage A's; Stage B is a different task)
torch.manual_seed(0); np.random.seed(0)
model_b = GreyBoxPINN().to(DEVICE)
history_b = train_stage_b(model_b, train_windows, val_windows, NOISE_PARAMS_B)

# Persist Stage B artifacts
save_model_and_history(model_b, history_b, 'stage_b')


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Drift vs epoch with classical DR baseline
axes[0].plot(history_b['epoch'], history_b['val_drift_30s'], 'b-o', label='PINN')
axes[0].axhline(np.mean(sample_drifts), color='r', ls='--', label='Classical DR mean')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('val drift @ 30s (m)')
axes[0].legend(); axes[0].grid(alpha=0.3); axes[0].set_title('Drift vs DR')

# Bias correlation over training
axes[1].plot(history_b['epoch'], history_b['bias_corr_a'], 'C0-o', label='b_a')
axes[1].plot(history_b['epoch'], history_b['bias_corr_w'], 'C1-o', label='b_w')
axes[1].axhline(0.7, color='gray', ls=':', alpha=0.5, label='good (0.7)')
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('correlation')
axes[1].set_ylim(-0.1, 1.05); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_title('Bias estimation quality')

# True vs predicted bias scatter (final model)
model_b.eval()
true_a, pred_a, true_w, pred_w = [], [], [], []
with torch.no_grad():
    for vw in val_windows:
        for s in range(10):
            ax_v, ay_v, wz_v, dt_v, v0_v, yaw0_v, b_a_t, b_w_t = \
                stack_batch_noisy([vw], NOISE_PARAMS_B, base_seed=2_000_000 + s * 100)
            _, _, _, _, bias_v = model_b(ax_v, ay_v, wz_v, dt_v, v0_v, yaw0_v)
            true_a.append(float(b_a_t[0])); pred_a.append(float(bias_v[:,:,0].mean()))
            true_w.append(float(b_w_t[0])); pred_w.append(float(bias_v[:,:,1].mean()))

axes[2].scatter(true_a, pred_a, alpha=0.5)
lim = max(abs(np.array(true_a)).max(), abs(np.array(pred_a)).max()) * 1.1
axes[2].plot([-lim, lim], [-lim, lim], 'k--', alpha=0.5)
axes[2].set_xlabel('true b_a (m/s²)'); axes[2].set_ylabel('predicted b_a')
axes[2].set_title(f'Accel bias (r={np.corrcoef(true_a, pred_a)[0,1]:.2f})')
axes[2].grid(alpha=0.3)

axes[3].scatter(true_w, pred_w, alpha=0.5)
lim = max(abs(np.array(true_w)).max(), abs(np.array(pred_w)).max()) * 1.1
axes[3].plot([-lim, lim], [-lim, lim], 'k--', alpha=0.5)
axes[3].set_xlabel('true b_w (rad/s)'); axes[3].set_ylabel('predicted b_w')
axes[3].set_title(f'Gyro bias (r={np.corrcoef(true_w, pred_w)[0,1]:.2f})')
axes[3].grid(alpha=0.3)

plt.tight_layout(); save_current_fig('stage_b_sanity_5a_panels'); plt.show()

## models comparison plots

In [ ]:
def plot_trajectory_comparison(model, val_windows, noise_params, n_show=6,
                                base_seed=5_000_000, title=None):
    """
    Show classical DR vs PINN trajectories on a sample of val windows.
    Same noisy IMU fed to both, so comparison is apples-to-apples.
    """
    model.eval()
    n_cols = 3
    n_rows = (n_show + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4.5*n_rows))
    axes = np.atleast_2d(axes).flatten()

    drifts_dr, drifts_pinn = [], []

    for i, vw in enumerate(val_windows[:n_show]):
        # Inject the SAME noisy IMU once, run both models on it
        noisy = inject_constant_bias(vw, noise_params, seed=base_seed + i)
        # Build noisy window for PINN
        ax_v = torch.tensor(noisy['ax'][None],   dtype=torch.float32, device=DEVICE)
        ay_v = torch.tensor(vw['ay'][None],      dtype=torch.float32, device=DEVICE)
        wz_v = torch.tensor(noisy['wz'][None],   dtype=torch.float32, device=DEVICE)
        dt_v = torch.tensor(vw['dt'][None],      dtype=torch.float32, device=DEVICE)
        v0_v = torch.tensor([vw['v0']],          dtype=torch.float32, device=DEVICE)
        yaw0_v = torch.tensor([vw['yaw0']],      dtype=torch.float32, device=DEVICE)

        with torch.no_grad():
            x_p, y_p, _, _, _ = model(ax_v, ay_v, wz_v, dt_v, v0_v, yaw0_v)
        x_p = x_p[0].cpu().numpy(); y_p = y_p[0].cpu().numpy()

        # Classical DR on the same noisy IMU
        x_dr, y_dr = classical_dr(vw, noisy)

        # Final drift
        drift_dr   = np.sqrt((x_dr[-1]-vw['x_gt'][-1])**2 + (y_dr[-1]-vw['y_gt'][-1])**2)
        drift_pinn = np.sqrt((x_p[-1]-vw['x_gt'][-1])**2  + (y_p[-1]-vw['y_gt'][-1])**2)
        drifts_dr.append(drift_dr); drifts_pinn.append(drift_pinn)

        ax = axes[i]
        ax.plot(vw['x_gt'], vw['y_gt'], 'k-',  lw=2.5, label='Ground truth', zorder=3)
        ax.plot(x_dr, y_dr,             'r--', lw=1.5,
                label=f'Classical DR (drift={drift_dr:.1f}m)')
        ax.plot(x_p,  y_p,              'b-',  lw=1.5,
                label=f'PINN (drift={drift_pinn:.1f}m)')
        ax.plot(vw['x_gt'][0],  vw['y_gt'][0],  'go', ms=8, zorder=4)
        ax.plot(vw['x_gt'][-1], vw['y_gt'][-1], 'k*', ms=12, zorder=4)
        ax.set_aspect('equal'); ax.grid(alpha=0.3); ax.legend(fontsize=8, loc='best')
        ax.set_title(f"{vw['drive']}  v0={vw['v0']:.1f}")

    # Hide unused axes
    for j in range(n_show, len(axes)):
        axes[j].axis('off')

    if title:
        fig.suptitle(title, fontsize=14, y=1.02)
    plt.tight_layout(); save_current_fig('plot_traj_comparison_def'); plt.show()

    print(f"\nSummary across {n_show} windows:")
    print(f"  DR mean drift:   {np.mean(drifts_dr):6.2f} m")
    print(f"  PINN mean drift: {np.mean(drifts_pinn):6.2f} m")
    print(f"  Improvement:     {(1 - np.mean(drifts_pinn)/np.mean(drifts_dr))*100:+.1f}%")

In [ ]:
def plot_trajectory_comparison_clean(model, val_windows, n_show=6, title=None):
    """For Stage A: clean IMU, no noise injection. DR should match GT exactly."""
    model.eval()
    n_cols = 3; n_rows = (n_show + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4.5*n_rows))
    axes = np.atleast_2d(axes).flatten()

    for i, vw in enumerate(val_windows[:n_show]):
        ax_v, ay_v, wz_v, dt_v, v0_v, yaw0_v = stack_batch([vw])
        with torch.no_grad():
            x_p, y_p, _, _, _ = model(ax_v, ay_v, wz_v, dt_v, v0_v, yaw0_v)
        x_p = x_p[0].cpu().numpy(); y_p = y_p[0].cpu().numpy()

        # Classical DR with clean IMU = GT (this is how the data was generated)
        ax = axes[i]
        ax.plot(vw['x_gt'], vw['y_gt'], 'k-',  lw=2.5, label='GT (= clean DR)', zorder=3)
        ax.plot(x_p, y_p,                'b--', lw=1.5, label='PINN')
        ax.plot(vw['x_gt'][0],  vw['y_gt'][0],  'go', ms=8, zorder=4)
        ax.plot(vw['x_gt'][-1], vw['y_gt'][-1], 'k*', ms=12, zorder=4)
        ax.set_aspect('equal'); ax.grid(alpha=0.3); ax.legend(fontsize=8)
        drift = np.sqrt((x_p[-1]-vw['x_gt'][-1])**2 + (y_p[-1]-vw['y_gt'][-1])**2)
        ax.set_title(f"v0={vw['v0']:.1f}  drift={drift:.3f}m")
    for j in range(n_show, len(axes)): axes[j].axis('off')
    if title: fig.suptitle(title, fontsize=14, y=1.02)
    plt.tight_layout(); save_current_fig('plot_traj_comparison_clean_def'); plt.show()

In [ ]:
# Stage A model (clean IMU)
plot_trajectory_comparison_clean(model_a, val_windows,
                                 title='Stage A: clean IMU')

# Stage B model (constant bias)
plot_trajectory_comparison(model_b, val_windows, NOISE_PARAMS_B,
                           title='Stage B: constant bias only')

## constant bias + Random walk

In [ ]:
# --- 1. New noise parameters for Stage C ---
NOISE_PARAMS_C = dict(
    sigma_b0_a=0.10,       # constant bias, same as B
    sigma_b0_w=0.015,
    sigma_a=0.05,          # NEW: white noise (m/s²)
    sigma_w=0.01,          # NEW: white noise (rad/s)
    sigma_brwa=0.001,      # NEW: random walk (m/s²/√s)
    sigma_brww=0.0002,     # NEW: random walk (rad/s/√s)
)

# --- 2. New noise injection function ---
def inject_full_noise(window, noise_params, seed):
    """
    Apply the full noise model: constant bias + white noise + bias random walk.
    Returns dict with noisy IMU plus diagnostics.
    """
    rng = np.random.default_rng(seed)
    N = len(window['t'])
    dt = float(window['dt'][0])  # assume uniform within a window

    # Constant bias (one number per window)
    b0_ax = rng.normal(0.0, noise_params['sigma_b0_a'])
    b0_wz = rng.normal(0.0, noise_params['sigma_b0_w'])

    # White noise (per timestep)
    white_ax = rng.normal(0.0, noise_params['sigma_a'], N)
    white_wz = rng.normal(0.0, noise_params['sigma_w'], N)

    # Bias random walk (cumulative sum of small Gaussian increments)
    brw_ax = np.cumsum(rng.normal(0.0, noise_params['sigma_brwa']*np.sqrt(dt), N))
    brw_wz = np.cumsum(rng.normal(0.0, noise_params['sigma_brww']*np.sqrt(dt), N))

    # Total bias profile (per timestep — varies because of random walk)
    bias_a_profile = b0_ax + brw_ax
    bias_w_profile = b0_wz + brw_wz

    return {
        'ax':              window['ax'] + bias_a_profile + white_ax,
        'ay':              window['ay'],
        'wz':              window['wz'] + bias_w_profile + white_wz,
        'bias_a_profile':  bias_a_profile,   # per-timestep ground truth bias
        'bias_w_profile':  bias_w_profile,
        'b0_ax': float(b0_ax), 'b0_wz': float(b0_wz),
    }


# --- 3. Updated stack_batch_noisy ---
def stack_batch_noisy(windows, noise_params, base_seed):
    noisy = [inject_full_noise(w, noise_params, base_seed + i)
             for i, w in enumerate(windows)]
    ax = torch.tensor(np.stack([n['ax'] for n in noisy]),
                      dtype=torch.float32, device=DEVICE)
    ay = torch.tensor(np.stack([n['ay'] for n in noisy]),
                      dtype=torch.float32, device=DEVICE)
    wz = torch.tensor(np.stack([n['wz'] for n in noisy]),
                      dtype=torch.float32, device=DEVICE)
    dt = torch.tensor(np.stack([w['dt'] for w in windows]),
                      dtype=torch.float32, device=DEVICE)
    v0 = torch.tensor([w['v0']   for w in windows],
                      dtype=torch.float32, device=DEVICE)
    yaw0 = torch.tensor([w['yaw0'] for w in windows],
                        dtype=torch.float32, device=DEVICE)
    # Now we report the MEAN of the per-timestep bias as our scalar diagnostic.
    # This is what the corr metric will compare against.
    b_a_true = torch.tensor([n['bias_a_profile'].mean() for n in noisy],
                            dtype=torch.float32, device=DEVICE)
    b_w_true = torch.tensor([n['bias_w_profile'].mean() for n in noisy],
                            dtype=torch.float32, device=DEVICE)
    return ax, ay, wz, dt, v0, yaw0, b_a_true, b_w_true

In [ ]:
np.random.seed(42)
sample_drifts = []
for vw in val_windows[:10]:
    for s in range(5):
        noisy = inject_full_noise(vw, NOISE_PARAMS_C, seed=2000 + s)
        x_dr, y_dr = classical_dr(vw, noisy)
        drift = np.sqrt((x_dr[-1] - vw['x_gt'][-1])**2 +
                        (y_dr[-1] - vw['y_gt'][-1])**2)
        sample_drifts.append(drift)
print(f"Stage C classical DR drift on val:")
print(f"  mean: {np.mean(sample_drifts):.2f} m,  median: {np.median(sample_drifts):.2f} m")

In [ ]:
torch.manual_seed(0); np.random.seed(0)
model_c_smooth = GreyBoxPINN().to(DEVICE)
history_c_smooth = train_stage_b(model_c_smooth, train_windows, val_windows,
                                 NOISE_PARAMS_C, lambda_smooth=100.0)

# Persist Stage C smooth artifacts
save_model_and_history(model_c_smooth, history_c_smooth, 'stage_c_smooth')


In [ ]:
def evaluate_and_visualize(model, val_windows, noise_params=None,
                          stage_name=''):
    """Standard post-training evaluation: trajectory comparison + summary."""
    if noise_params is None:
        plot_trajectory_comparison_clean(model, val_windows,
                                         title=f'{stage_name}: trajectories')
    else:
        plot_trajectory_comparison(model, val_windows, noise_params,
                                   title=f'{stage_name}: trajectories')

# After every training:
evaluate_and_visualize(model_c_smooth, val_windows, NOISE_PARAMS_C,
                       stage_name='Stage C')

In [ ]:
import matplotlib.pyplot as plt

model_c_smooth.eval()
fig, axes = plt.subplots(2, 4, figsize=(18, 7))

for col, vw in enumerate(val_windows[:4]):
    noisy = inject_full_noise(vw, NOISE_PARAMS_C, seed=7_000_000 + col)
    ax_v = torch.tensor(noisy['ax'][None],   dtype=torch.float32, device=DEVICE)
    ay_v = torch.tensor(vw['ay'][None],      dtype=torch.float32, device=DEVICE)
    wz_v = torch.tensor(noisy['wz'][None],   dtype=torch.float32, device=DEVICE)
    dt_v = torch.tensor(vw['dt'][None],      dtype=torch.float32, device=DEVICE)
    v0_v = torch.tensor([vw['v0']],          dtype=torch.float32, device=DEVICE)
    yaw0_v = torch.tensor([vw['yaw0']],      dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        _, _, _, _, bias_p = model_c_smooth(ax_v, ay_v, wz_v, dt_v, v0_v, yaw0_v)
    pred_a = bias_p[0, :, 0].cpu().numpy()
    pred_w = bias_p[0, :, 1].cpu().numpy()
    t = vw['t']

    axes[0, col].plot(t, noisy['bias_a_profile'], 'k-', lw=2, label='true bias')
    axes[0, col].plot(t, pred_a,                  'b-', lw=1.5, label='predicted')
    axes[0, col].axhline(noisy['b0_ax'], color='gray', ls=':', label='constant b₀')
    axes[0, col].set_title(f'Accel bias (window {col})')
    axes[0, col].grid(alpha=0.3); axes[0, col].legend(fontsize=8)
    axes[0, col].set_xlabel('t (s)'); axes[0, col].set_ylabel('m/s²')

    axes[1, col].plot(t, noisy['bias_w_profile'], 'k-', lw=2, label='true bias')
    axes[1, col].plot(t, pred_w,                  'b-', lw=1.5, label='predicted')
    axes[1, col].axhline(noisy['b0_wz'], color='gray', ls=':', label='constant b₀')
    axes[1, col].set_title(f'Gyro bias (window {col})')
    axes[1, col].grid(alpha=0.3); axes[1, col].legend(fontsize=8)
    axes[1, col].set_xlabel('t (s)'); axes[1, col].set_ylabel('rad/s')

plt.tight_layout(); save_current_fig('per_timestep_bias_inspection'); plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Pick one window for visualization
w = generate_synthetic_window(motion='circle',
                              motion_params={'wz': 0.05},
                              v0=10.0, yaw0=0.0, seed=1)

# Inject the full noise model (constant bias + white noise + random walk)
noisy = inject_full_noise(w, NOISE_PARAMS_C, seed=42)

t = w['t']
fig, axes = plt.subplots(2, 3, figsize=(18, 8))

# --- Top row: ax ---
axes[0, 0].plot(t, w['ax'], 'k-', lw=2, label='clean (true)')
axes[0, 0].set_title('Clean ax (what the IMU should read)')
axes[0, 0].grid(alpha=0.3); axes[0, 0].legend(); axes[0, 0].set_ylabel('m/s²')

axes[0, 1].plot(t, noisy['bias_a_profile'], 'C2-', lw=2, label='true bias profile')
axes[0, 1].axhline(noisy['b0_ax'], color='gray', ls=':', label=f'constant b₀={noisy["b0_ax"]:+.3f}')
axes[0, 1].set_title('True bias = constant + random walk')
axes[0, 1].grid(alpha=0.3); axes[0, 1].legend()

axes[0, 2].plot(t, w['ax'],     'k-', lw=2, alpha=0.6, label='clean')
axes[0, 2].plot(t, noisy['ax'], 'r-', lw=0.8, label='noisy (input to model)')
axes[0, 2].set_title('Clean vs noisy ax')
axes[0, 2].grid(alpha=0.3); axes[0, 2].legend()

# --- Bottom row: wz ---
axes[1, 0].plot(t, w['wz'], 'k-', lw=2, label='clean (true)')
axes[1, 0].set_title('Clean wz')
axes[1, 0].grid(alpha=0.3); axes[1, 0].legend(); axes[1, 0].set_ylabel('rad/s')
axes[1, 0].set_xlabel('t (s)')

axes[1, 1].plot(t, noisy['bias_w_profile'], 'C2-', lw=2, label='true bias profile')
axes[1, 1].axhline(noisy['b0_wz'], color='gray', ls=':', label=f'constant b₀={noisy["b0_wz"]:+.4f}')
axes[1, 1].set_title('True bias = constant + random walk')
axes[1, 1].grid(alpha=0.3); axes[1, 1].legend()
axes[1, 1].set_xlabel('t (s)')

axes[1, 2].plot(t, w['wz'],     'k-', lw=2, alpha=0.6, label='clean')
axes[1, 2].plot(t, noisy['wz'], 'r-', lw=0.8, label='noisy (input to model)')
axes[1, 2].set_title('Clean vs noisy wz')
axes[1, 2].grid(alpha=0.3); axes[1, 2].legend()
axes[1, 2].set_xlabel('t (s)')

plt.tight_layout(); save_current_fig('noisy_imu_visualization'); plt.show()

# Print a magnitude summary
print(f"\nNoise component magnitudes (this window, this seed):")
print(f"  Constant bias    b0_ax={noisy['b0_ax']:+.4f} m/s²    b0_wz={noisy['b0_wz']:+.4f} rad/s")
print(f"  RW final value   {noisy['bias_a_profile'][-1] - noisy['b0_ax']:+.4f}            "
      f"{noisy['bias_w_profile'][-1] - noisy['b0_wz']:+.5f}")
print(f"  Total bias range [{noisy['bias_a_profile'].min():+.3f}, {noisy['bias_a_profile'].max():+.3f}]   "
      f"[{noisy['bias_w_profile'].min():+.4f}, {noisy['bias_w_profile'].max():+.4f}]")
print(f"  Clean ax range   [{w['ax'].min():+.3f}, {w['ax'].max():+.3f}]              "
      f"[{w['wz'].min():+.4f}, {w['wz'].max():+.4f}]")

## Stage C: bis


In [ ]:
# --- Fixed eval set: same windows, same noise seeds, used for ALL models ---

EVAL_VAL_WINDOWS = make_dataset(100, seed_base=10_000)   # the 100 hardest val test
EVAL_NOISE_SEEDS = list(range(900_000, 900_010))         # 10 noise realizations per window

def evaluate_model(model, val_windows, noise_params, noise_seeds):
    """Compute the 5 comparison metrics on a fixed eval set."""
    model.eval()
    drifts, true_bas, pred_bas, true_bws, pred_bws = [], [], [], [], []
    smooth_vals, tracking_a, tracking_w = [], [], []

    with torch.no_grad():
        for vw in val_windows:
            for s in noise_seeds:
                noisy = inject_full_noise(vw, noise_params, seed=s)
                ax_v = torch.tensor(noisy['ax'][None], dtype=torch.float32, device=DEVICE)
                ay_v = torch.tensor(vw['ay'][None],    dtype=torch.float32, device=DEVICE)
                wz_v = torch.tensor(noisy['wz'][None], dtype=torch.float32, device=DEVICE)
                dt_v = torch.tensor(vw['dt'][None],    dtype=torch.float32, device=DEVICE)
                v0_v = torch.tensor([vw['v0']],        dtype=torch.float32, device=DEVICE)
                yaw0_v = torch.tensor([vw['yaw0']],    dtype=torch.float32, device=DEVICE)
                x_v, y_v, _, _, bias_v = model(ax_v, ay_v, wz_v, dt_v, v0_v, yaw0_v)

                drift = float(torch.sqrt(
                    (x_v[0, -1] - vw['x_gt'][-1])**2 +
                    (y_v[0, -1] - vw['y_gt'][-1])**2))
                drifts.append(drift)

                # Window-mean bias for correlation
                true_bas.append(noisy['bias_a_profile'].mean())
                pred_bas.append(float(bias_v[:,:,0].mean()))
                true_bws.append(noisy['bias_w_profile'].mean())
                pred_bws.append(float(bias_v[:,:,1].mean()))

                # Smoothness: sum of squared per-step deltas of predicted bias
                bp = bias_v[0].cpu().numpy()    # (T, 2)
                smooth_vals.append(float(np.mean(np.diff(bp, axis=0)**2)))

                # Per-timestep tracking error (NOT averaged before comparing)
                pred_a_full = bias_v[0, :, 0].cpu().numpy()
                pred_w_full = bias_v[0, :, 1].cpu().numpy()
                tracking_a.append(float(np.mean((pred_a_full - noisy['bias_a_profile'])**2)))
                tracking_w.append(float(np.mean((pred_w_full - noisy['bias_w_profile'])**2)))

    return {
        'drift_mean':   float(np.mean(drifts)),
        'drift_median': float(np.median(drifts)),
        'drift_p95':    float(np.percentile(drifts, 95)),
        'corr_a':       float(np.corrcoef(true_bas, pred_bas)[0, 1]),
        'corr_w':       float(np.corrcoef(true_bws, pred_bws)[0, 1]),
        'smoothness':   float(np.mean(smooth_vals)),
        'tracking_a':   float(np.mean(tracking_a)),
        'tracking_w':   float(np.mean(tracking_w)),
        'all_drifts':   drifts,
    }

In [ ]:
# === Stage C-bis: comparative analysis grid (unattended-safe) ===
# Saves models + results + history to /kaggle/working incrementally.
# Safe to re-run: skips already-saved models. Won't crash on individual failures.

import os
import pickle
import json
import traceback
import torch
import numpy as np

LAMBDAS    = [0.0, 1_000.0, 100_000.0]
DATA_SIZES = [(80, 20), (200, 50), (500, 100)]
NOISE_PARAMS = NOISE_PARAMS_C
N_EPOCHS = 600

SAVE_DIR = '/kaggle/working/stage_c_bis'
os.makedirs(SAVE_DIR, exist_ok=True)

results = {}
trained_models = {}
training_histories = {}

# Reload anything previously saved (resumability)
results_path = f'{SAVE_DIR}/results.pkl'
if os.path.exists(results_path):
    try:
        with open(results_path, 'rb') as f:
            results = pickle.load(f)
        print(f"Loaded {len(results)} existing results.")
    except Exception as e:
        print(f"Warning: could not load existing results: {e}")
        results = {}

def save_progress():
    """Write everything to disk. Idempotent."""
    try:
        with open(results_path, 'wb') as f:
            pickle.dump(results, f)
        # Also save a human-readable summary
        with open(f'{SAVE_DIR}/results_summary.json', 'w') as f:
            summary = {f"lam{k[0]:g}_n{k[1]}": {
                kk: vv for kk, vv in v.items() if kk != 'all_drifts'
            } for k, v in results.items()}
            json.dump(summary, f, indent=2)
    except Exception as e:
        print(f"  WARN: save_progress failed: {e}")

# Main grid
total_models = len(LAMBDAS) * len(DATA_SIZES)
done = 0

for train_n, val_n in DATA_SIZES:
    try:
        train_set = make_dataset(train_n, seed_base=0)
        val_set   = make_dataset(val_n,   seed_base=10_000)
    except Exception as e:
        print(f"\n!! Could not build datasets for n={train_n}: {e}")
        traceback.print_exc()
        continue

    for lam in LAMBDAS:
        done += 1
        key = (lam, train_n)
        model_path   = f'{SAVE_DIR}/model_lam{lam:g}_n{train_n}.pt'
        history_path = f'{SAVE_DIR}/history_lam{lam:g}_n{train_n}.pkl'

        # Skip if already done in a previous run
        if key in results and os.path.exists(model_path):
            print(f"\n[{done}/{total_models}] λ={lam:g}, n={train_n}: SKIP (already saved)")
            # Still load the model into memory in case downstream cells use it
            try:
                model = GreyBoxPINN().to(DEVICE)
                model.load_state_dict(torch.load(model_path, map_location=DEVICE))
                trained_models[key] = model
                if os.path.exists(history_path):
                    with open(history_path, 'rb') as f:
                        training_histories[key] = pickle.load(f)
            except Exception as e:
                print(f"  WARN: could not reload {key}: {e}")
            continue

        print(f"\n{'='*60}")
        print(f"[{done}/{total_models}] λ={lam:>10g}, train_size={train_n}")
        print(f"{'='*60}")

        try:
            torch.manual_seed(0); np.random.seed(0)
            model = GreyBoxPINN().to(DEVICE)
            history = train_stage_b(model, train_set, val_set, NOISE_PARAMS,
                                    n_epochs=N_EPOCHS, lambda_smooth=lam,
                                    log_every=100)

            # Evaluate on the FIXED eval set
            metrics = evaluate_model(model, EVAL_VAL_WINDOWS,
                                     NOISE_PARAMS, EVAL_NOISE_SEEDS)

            # Save everything immediately
            torch.save(model.state_dict(), model_path)
            with open(history_path, 'wb') as f:
                pickle.dump(history, f)

            results[key] = metrics
            trained_models[key] = model
            training_histories[key] = history

            save_progress()

            print(f"  -> drift_mean={metrics['drift_mean']:.2f}m  "
                  f"smoothness={metrics['smoothness']:.6f}  "
                  f"tracking_a={metrics['tracking_a']:.5f}")
            print(f"  -> saved: {model_path}")

        except Exception as e:
            print(f"\n  !! Training failed for λ={lam}, n={train_n}: {e}")
            traceback.print_exc()
            # Keep going; record the failure
            results[key] = {'error': str(e), 'drift_mean': float('inf')}
            save_progress()
            continue

print(f"\n{'='*60}")
print(f"Grid complete. {len(trained_models)} models trained, "
      f"{len(results)} results recorded.")
print(f"All artifacts in: {SAVE_DIR}")
print(f"{'='*60}")

In [ ]:
# Heatmap: val drift / smoothness / tracking across the (lambda, dataset) grid.
import matplotlib.pyplot as plt

if results and trained_models:
    # Determine grid axes from results so the plot reflects what was actually trained.
    lambdas_sorted = sorted({k[0] for k in results.keys() if 'error' not in results[k]})
    sizes_sorted   = sorted({k[1] for k in results.keys() if 'error' not in results[k]})

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    metric_specs = [
        (axes[0], 'drift_mean', 'Val drift mean (m) - lower is better', '.1f'),
        (axes[1], 'smoothness', 'Bias smoothness (lower = less wobble)', '.5f'),
        (axes[2], 'tracking_a', 'Accel bias tracking error (lower = better)', '.4f'),
    ]
    for ax, metric, title, fmt in metric_specs:
        grid = np.full((len(lambdas_sorted), len(sizes_sorted)), np.nan)
        for i, lam in enumerate(lambdas_sorted):
            for j, n in enumerate(sizes_sorted):
                if (lam, n) in results and 'error' not in results[(lam, n)]:
                    grid[i, j] = results[(lam, n)][metric]
        im = ax.imshow(grid, aspect='auto', cmap='RdYlGn_r')
        ax.set_xticks(range(len(sizes_sorted)))
        ax.set_xticklabels([str(n) for n in sizes_sorted])
        ax.set_yticks(range(len(lambdas_sorted)))
        ax.set_yticklabels([f"λ={lam:g}" for lam in lambdas_sorted])
        ax.set_xlabel('train dataset size')
        ax.set_title(title)
        for i in range(len(lambdas_sorted)):
            for j in range(len(sizes_sorted)):
                if not np.isnan(grid[i, j]):
                    ax.text(j, i, f"{grid[i, j]:{fmt}}", ha='center', va='center',
                            color='black', fontsize=10)
        plt.colorbar(im, ax=ax, fraction=0.046)

    plt.tight_layout()
    save_current_fig('grid_heatmap')
    plt.show()
else:
    print("Skipping heatmap: no results available.")


In [ ]:
# Best vs worst model: per-timestep bias profile on a single window.
if results and trained_models:
    # Pick best by drift_mean, worst by tracking error
    valid_keys = [k for k, v in results.items() if 'error' not in v]
    if valid_keys:
        best_key  = min(valid_keys, key=lambda k: results[k]['drift_mean'])
        worst_key = max(valid_keys, key=lambda k: results[k]['tracking_a'])
        best_model  = trained_models.get(best_key)
        worst_model = trained_models.get(worst_key)

        if best_model is not None and worst_model is not None:
            print(f"Best:  λ={best_key[0]:g}, n={best_key[1]}  "
                  f"drift_mean={results[best_key]['drift_mean']:.2f}m")
            print(f"Worst: λ={worst_key[0]:g}, n={worst_key[1]}  "
                  f"tracking_a={results[worst_key]['tracking_a']:.5f}")

            # Trajectory comparison vs DR for best model
            evaluate_and_visualize(best_model, EVAL_VAL_WINDOWS[:6], NOISE_PARAMS_C,
                                   stage_name=f'Best model (λ={best_key[0]:g}, n={best_key[1]})')

            # Per-timestep bias for best vs worst on the same window
            vw = EVAL_VAL_WINDOWS[0]
            noisy = inject_full_noise(vw, NOISE_PARAMS_C, seed=7_000_000)

            fig, axes = plt.subplots(2, 2, figsize=(14, 8))
            for col, (model, label, key) in enumerate([
                (best_model,  'Best',  best_key),
                (worst_model, 'Worst', worst_key),
            ]):
                ax_v = torch.tensor(noisy['ax'][None], dtype=torch.float32, device=DEVICE)
                ay_v = torch.tensor(vw['ay'][None],    dtype=torch.float32, device=DEVICE)
                wz_v = torch.tensor(noisy['wz'][None], dtype=torch.float32, device=DEVICE)
                dt_v = torch.tensor(vw['dt'][None],    dtype=torch.float32, device=DEVICE)
                v0_v = torch.tensor([vw['v0']],        dtype=torch.float32, device=DEVICE)
                yaw0_v = torch.tensor([vw['yaw0']],    dtype=torch.float32, device=DEVICE)
                with torch.no_grad():
                    _, _, _, _, bias_p = model(ax_v, ay_v, wz_v, dt_v, v0_v, yaw0_v)

                axes[0, col].plot(vw['t'], noisy['bias_a_profile'], 'k-', lw=2, label='true')
                axes[0, col].plot(vw['t'], bias_p[0,:,0].cpu().numpy(), 'b-', lw=1.5, label='predicted')
                axes[0, col].set_title(f"{label} model (λ={key[0]:g}, n={key[1]}) - accel bias")
                axes[0, col].legend(); axes[0, col].grid(alpha=0.3)

                axes[1, col].plot(vw['t'], noisy['bias_w_profile'], 'k-', lw=2, label='true')
                axes[1, col].plot(vw['t'], bias_p[0,:,1].cpu().numpy(), 'b-', lw=1.5, label='predicted')
                axes[1, col].set_title(f"{label} model - gyro bias")
                axes[1, col].legend(); axes[1, col].grid(alpha=0.3)

            plt.tight_layout()
            save_current_fig('best_vs_worst_bias')
            plt.show()
        else:
            print("Skipping best/worst plot: best or worst model not in memory.")
    else:
        print("Skipping best/worst plot: no successful results.")
else:
    print("Skipping best/worst plot: no results available.")


In [ ]:
def best_model_comparison(results, trained_models, val_windows, noise_params,
                          n_windows=4, base_seed=8_000_000):
    """
    For each dataset size, pick the best model (lowest drift_mean),
    then visualize its trajectories alongside DR and GT on n_windows
    sample val windows.
    """
    # 1. Find best model per dataset size
    dataset_sizes = sorted({n for (_, n) in results.keys()})
    best_per_size = {}
    for n in dataset_sizes:
        candidates = {k: v for k, v in results.items() if k[1] == n}
        best_key = min(candidates.keys(), key=lambda k: candidates[k]['drift_mean'])
        best_per_size[n] = best_key

    print("Best model per dataset size (by drift_mean):")
    for n, key in best_per_size.items():
        m = results[key]
        print(f"  n={n:4d}: λ={key[0]:>10g}  "
              f"drift_mean={m['drift_mean']:.2f}m  "
              f"smoothness={m['smoothness']:.6f}  "
              f"tracking_a={m['tracking_a']:.5f}")

    # 2. Set up the grid
    fig, axes = plt.subplots(n_windows, len(dataset_sizes),
                             figsize=(5 * len(dataset_sizes), 4.5 * n_windows),
                             squeeze=False)

    # Color palette for the PINNs (distinguishable from each other)
    pinn_colors = {80: 'tab:blue', 200: 'tab:purple', 500: 'tab:green'}

    # Pre-compute DR and PINN trajectories for each (window, model)
    for row, vw in enumerate(val_windows[:n_windows]):
        # Same noise injection for all models on this window — apples to apples
        noisy = inject_full_noise(vw, noise_params, seed=base_seed + row)
        # Classical DR
        x_dr, y_dr = classical_dr(vw, noisy)
        drift_dr = np.sqrt((x_dr[-1] - vw['x_gt'][-1])**2 +
                          (y_dr[-1] - vw['y_gt'][-1])**2)

        for col, n in enumerate(dataset_sizes):
            ax = axes[row, col]
            best_key = best_per_size[n]
            model = trained_models[best_key]

            # Run PINN on the same noisy IMU
            ax_v = torch.tensor(noisy['ax'][None], dtype=torch.float32, device=DEVICE)
            ay_v = torch.tensor(vw['ay'][None],    dtype=torch.float32, device=DEVICE)
            wz_v = torch.tensor(noisy['wz'][None], dtype=torch.float32, device=DEVICE)
            dt_v = torch.tensor(vw['dt'][None],    dtype=torch.float32, device=DEVICE)
            v0_v = torch.tensor([vw['v0']],        dtype=torch.float32, device=DEVICE)
            yaw0_v = torch.tensor([vw['yaw0']],    dtype=torch.float32, device=DEVICE)
            with torch.no_grad():
                x_p, y_p, _, _, _ = model(ax_v, ay_v, wz_v, dt_v, v0_v, yaw0_v)
            x_p, y_p = x_p[0].cpu().numpy(), y_p[0].cpu().numpy()
            drift_pinn = np.sqrt((x_p[-1] - vw['x_gt'][-1])**2 +
                                (y_p[-1] - vw['y_gt'][-1])**2)

            # Plot all three
            ax.plot(vw['x_gt'], vw['y_gt'], 'k-', lw=2.5,
                    label='Ground truth', zorder=3)
            ax.plot(x_dr, y_dr, 'r--', lw=1.5,
                    label=f'DR ({drift_dr:.1f}m)', alpha=0.8)
            ax.plot(x_p, y_p, '-', color=pinn_colors[n], lw=1.8,
                    label=f'PINN ({drift_pinn:.1f}m)')

            # Markers
            ax.plot(vw['x_gt'][0],  vw['y_gt'][0],  'go', ms=8, zorder=4)
            ax.plot(vw['x_gt'][-1], vw['y_gt'][-1], 'k*', ms=12, zorder=4)
            ax.set_aspect('equal')
            ax.grid(alpha=0.3)
            ax.legend(fontsize=8, loc='best')

            # Titles: top row gets the dataset label, all rows get window info
            if row == 0:
                ax.set_title(f"n={n}, λ={best_per_size[n][0]:g}\n"
                            f"{vw['drive']}  v0={vw['v0']:.1f}",
                            fontsize=10)
            else:
                ax.set_title(f"{vw['drive']}  v0={vw['v0']:.1f}", fontsize=10)

    plt.suptitle('Best PINN at each dataset size vs Classical DR',
                 fontsize=14, y=1.005)
    plt.tight_layout()
    save_current_fig('best_per_dataset')
    plt.show()


# Run it after the grid finishes (or even now with what you have)
if results and trained_models:
    best_model_comparison(results, trained_models, EVAL_VAL_WINDOWS,
                          NOISE_PARAMS_C, n_windows=4)
else:
    print('Skipping best_model_comparison: no results available.')

In [ ]:
def drift_distribution_per_size(results, trained_models, val_windows,
                                noise_params, base_seeds=range(900_000, 900_010)):
    """
    For the best model at each dataset size, compute drift on ALL eval
    windows × seeds and show the distribution as a violin plot.
    """
    dataset_sizes = sorted({n for (_, n) in results.keys()})
    best_per_size = {n: min({k: v for k, v in results.items() if k[1] == n}.keys(),
                            key=lambda k: results[k]['drift_mean'])
                    for n in dataset_sizes}

    # Compute DR drifts (same for all models)
    dr_drifts = []
    for vw in val_windows:
        for s in base_seeds:
            noisy = inject_full_noise(vw, noise_params, seed=s)
            x_dr, y_dr = classical_dr(vw, noisy)
            dr_drifts.append(np.sqrt((x_dr[-1]-vw['x_gt'][-1])**2 +
                                    (y_dr[-1]-vw['y_gt'][-1])**2))

    # Compute PINN drifts per best model
    all_drifts = {'DR': dr_drifts}
    for n in dataset_sizes:
        model = trained_models[best_per_size[n]]
        model.eval()
        drifts = []
        with torch.no_grad():
            for vw in val_windows:
                for s in base_seeds:
                    noisy = inject_full_noise(vw, noise_params, seed=s)
                    ax_v = torch.tensor(noisy['ax'][None], dtype=torch.float32, device=DEVICE)
                    ay_v = torch.tensor(vw['ay'][None],    dtype=torch.float32, device=DEVICE)
                    wz_v = torch.tensor(noisy['wz'][None], dtype=torch.float32, device=DEVICE)
                    dt_v = torch.tensor(vw['dt'][None],    dtype=torch.float32, device=DEVICE)
                    v0_v = torch.tensor([vw['v0']],        dtype=torch.float32, device=DEVICE)
                    yaw0_v = torch.tensor([vw['yaw0']],    dtype=torch.float32, device=DEVICE)
                    x_p, y_p, _, _, _ = model(ax_v, ay_v, wz_v, dt_v, v0_v, yaw0_v)
                    drifts.append(float(torch.sqrt((x_p[0,-1]-vw['x_gt'][-1])**2 +
                                                   (y_p[0,-1]-vw['y_gt'][-1])**2)))
        all_drifts[f'PINN n={n}'] = drifts

    # Plot
    fig, ax = plt.subplots(figsize=(10, 6))
    labels = list(all_drifts.keys())
    data = [all_drifts[l] for l in labels]

    parts = ax.violinplot(data, showmeans=False, showmedians=True, widths=0.7)
    for i, pc in enumerate(parts['bodies']):
        pc.set_facecolor(['lightcoral'] + ['lightblue', 'mediumpurple', 'lightgreen'][:len(labels)-1][i])
        pc.set_alpha(0.7)

    # Overlay mean as a marker
    means = [np.mean(d) for d in data]
    ax.scatter(range(1, len(labels)+1), means, color='black', s=50,
               zorder=3, label='mean', marker='D')

    ax.set_xticks(range(1, len(labels) + 1))
    ax.set_xticklabels(labels)
    ax.set_ylabel('Final drift (m)')
    ax.set_title(f'Drift distribution: DR vs best PINN at each dataset size\n'
                 f'({len(val_windows)} windows × {len(list(base_seeds))} noise seeds)')
    ax.grid(alpha=0.3, axis='y')
    ax.legend()

    print("Summary statistics:")
    print(f"{'Model':<15} {'mean':>8} {'median':>8} {'p95':>8}")
    for label, drifts in all_drifts.items():
        print(f"{label:<15} {np.mean(drifts):>7.2f}m  "
              f"{np.median(drifts):>7.2f}m  {np.percentile(drifts, 95):>7.2f}m")

    plt.tight_layout()
    save_current_fig('drift_distribution')
    plt.show()


if results and trained_models:
    drift_distribution_per_size(results, trained_models, EVAL_VAL_WINDOWS, NOISE_PARAMS_C)
else:
    print('Skipping drift_distribution_per_size: no results available.')

In [ ]:
def drift_by_motion_type(results, trained_models, val_windows, noise_params,
                        n_seeds=10, model_key=None):
    """
    Break down drift by motion type for DR and the best PINN.
    
    If model_key is None, picks the overall best model by drift_mean.
    Otherwise uses the specified (lambda, n) key.
    """
    # Pick the model
    if model_key is None:
        model_key = min(results.keys(), key=lambda k: results[k]['drift_mean'])
    model = trained_models[model_key]
    model.eval()
    print(f"Using model: λ={model_key[0]:g}, n={model_key[1]}  "
          f"(drift_mean={results[model_key]['drift_mean']:.2f}m)")

    # Group val windows by motion type
    by_motion = {}
    for vw in val_windows:
        # Drive name is e.g. 'synthetic_circle' — strip the prefix
        motion = vw['drive'].replace('synthetic_', '')
        by_motion.setdefault(motion, []).append(vw)

    print(f"\nMotion type counts in eval set:")
    for m, ws in sorted(by_motion.items()):
        print(f"  {m:20s}: {len(ws)} windows")

    # Compute drifts per motion type for DR and PINN
    dr_by_motion = {m: [] for m in by_motion}
    pinn_by_motion = {m: [] for m in by_motion}

    seeds = list(range(900_000, 900_000 + n_seeds))

    with torch.no_grad():
        for motion, windows in by_motion.items():
            for vw in windows:
                for s in seeds:
                    noisy = inject_full_noise(vw, noise_params, seed=s)
                    # DR
                    x_dr, y_dr = classical_dr(vw, noisy)
                    dr_by_motion[motion].append(
                        np.sqrt((x_dr[-1]-vw['x_gt'][-1])**2 +
                                (y_dr[-1]-vw['y_gt'][-1])**2))
                    # PINN
                    ax_v = torch.tensor(noisy['ax'][None], dtype=torch.float32, device=DEVICE)
                    ay_v = torch.tensor(vw['ay'][None],    dtype=torch.float32, device=DEVICE)
                    wz_v = torch.tensor(noisy['wz'][None], dtype=torch.float32, device=DEVICE)
                    dt_v = torch.tensor(vw['dt'][None],    dtype=torch.float32, device=DEVICE)
                    v0_v = torch.tensor([vw['v0']],        dtype=torch.float32, device=DEVICE)
                    yaw0_v = torch.tensor([vw['yaw0']],    dtype=torch.float32, device=DEVICE)
                    x_p, y_p, _, _, _ = model(ax_v, ay_v, wz_v, dt_v, v0_v, yaw0_v)
                    pinn_by_motion[motion].append(
                        float(torch.sqrt((x_p[0,-1]-vw['x_gt'][-1])**2 +
                                        (y_p[0,-1]-vw['y_gt'][-1])**2)))

    # ----- Plot: side-by-side bar chart with error bars -----
    motions = sorted(by_motion.keys())
    dr_means   = [np.mean(dr_by_motion[m])   for m in motions]
    dr_p95s    = [np.percentile(dr_by_motion[m], 95)   for m in motions]
    pinn_means = [np.mean(pinn_by_motion[m]) for m in motions]
    pinn_p95s  = [np.percentile(pinn_by_motion[m], 95) for m in motions]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Left: bar chart, mean drift per motion
    x = np.arange(len(motions))
    width = 0.35
    bars_dr   = axes[0].bar(x - width/2, dr_means,   width,
                            yerr=[np.zeros(len(motions)),
                                  np.array(dr_p95s) - np.array(dr_means)],
                            label='DR', color='lightcoral', capsize=4,
                            error_kw={'alpha': 0.6})
    bars_pinn = axes[0].bar(x + width/2, pinn_means, width,
                            yerr=[np.zeros(len(motions)),
                                  np.array(pinn_p95s) - np.array(pinn_means)],
                            label='PINN', color='steelblue', capsize=4,
                            error_kw={'alpha': 0.6})
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(motions, rotation=20, ha='right')
    axes[0].set_ylabel('Final drift (m)')
    axes[0].set_title('Mean drift per motion type (error bars to p95)')
    axes[0].legend()
    axes[0].grid(alpha=0.3, axis='y')

    # Annotate improvement % on each pair
    for i, m in enumerate(motions):
        improve = (1 - pinn_means[i]/dr_means[i]) * 100
        ymax = max(dr_p95s[i], pinn_p95s[i])
        axes[0].text(i, ymax * 1.05, f"{improve:+.0f}%",
                     ha='center', fontsize=9,
                     color='darkgreen' if improve > 0 else 'darkred')

    # Right: violin plot of the full distributions
    positions_dr   = np.arange(len(motions)) * 2.5
    positions_pinn = positions_dr + 0.9

    parts_dr   = axes[1].violinplot([dr_by_motion[m]   for m in motions],
                                    positions=positions_dr,   widths=0.8,
                                    showmeans=False, showmedians=True)
    parts_pinn = axes[1].violinplot([pinn_by_motion[m] for m in motions],
                                    positions=positions_pinn, widths=0.8,
                                    showmeans=False, showmedians=True)
    for pc in parts_dr['bodies']:   pc.set_facecolor('lightcoral'); pc.set_alpha(0.7)
    for pc in parts_pinn['bodies']: pc.set_facecolor('steelblue');  pc.set_alpha(0.7)

    axes[1].set_xticks(positions_dr + 0.45)
    axes[1].set_xticklabels(motions, rotation=20, ha='right')
    axes[1].set_ylabel('Final drift (m)')
    axes[1].set_title('Full drift distribution per motion type')
    axes[1].grid(alpha=0.3, axis='y')

    # Custom legend for violin plot
    from matplotlib.patches import Patch
    axes[1].legend(handles=[
        Patch(facecolor='lightcoral', alpha=0.7, label='DR'),
        Patch(facecolor='steelblue',  alpha=0.7, label='PINN'),
    ])

    plt.suptitle(f'Drift by motion type — best model '
                 f'(λ={model_key[0]:g}, n={model_key[1]})',
                 fontsize=13, y=1.02)
    plt.tight_layout()
    save_current_fig('drift_by_motion')
    plt.show()

    # Print summary
    print(f"\nPer-motion-type summary (mean drift, m):")
    print(f"{'Motion':<20} {'DR':>8} {'PINN':>8} {'Improvement':>14}")
    for i, m in enumerate(motions):
        improve = (1 - pinn_means[i]/dr_means[i]) * 100
        print(f"{m:<20} {dr_means[i]:>7.2f}  {pinn_means[i]:>7.2f}  "
              f"{improve:>+13.1f}%")

    return dr_by_motion, pinn_by_motion


# Run on the overall best model
if results and trained_models:
    dr_by_motion, pinn_by_motion = drift_by_motion_type(
        results, trained_models, EVAL_VAL_WINDOWS, NOISE_PARAMS_C)
else:
    print('Skipping drift_by_motion_type: no results available.')